# E-Commerce Product Analysis and Machine Learning

## Part 2: Classification and Machine Learning

This notebook focuses on building classification models to predict book rating categories using the cleaned Books to Scrape dataset.

### Objectives
- Prepare classification targets.
- Perform feature selection.
- Train classification algorithms.
- Evaluate and compare model performance.
- Perform cross-validation and hyperparameter tuning.
- Analyze feature importance.

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.impute import SimpleImputer

# loading cleaned dataset

In [2]:
df = pd.read_csv("../data/books_cleaned.csv")

print("Dataset loaded successfully!")
print("Dataset Shape:", df.shape)

df.head()

FileNotFoundError: [Errno 2] No such file or directory: '../data/books_cleaned.csv'

# inspecting dataset

In [ ]:
print("Column Names:")
print(df.columns.tolist())

print("\nDataset Information:")
df.info()

print("\nStatistical Summary:")
display(df.describe(include="all"))

print("\nMissing Values:")
print(df.isnull().sum())

## 2. Classification Target Preparation

The original star rating is converted into three categories:

- Low: 1 or 2 stars
- Medium: 3 stars
- High: 4 or 5 stars

This transforms the problem into a multiclass classification task.

In [ ]:
df_ml = df.copy()

def assign_rating_class(rating):
    if rating <= 2:
        return "Low"
    elif rating == 3:
        return "Medium"
    else:
        return "High"

df_ml["rating_class"] = df_ml["star_rating"].apply(assign_rating_class)

df_ml[["star_rating", "rating_class"]].head(10)

# checking class distribution

In [ ]:
print("Rating Class Distribution:")
print(df_ml["rating_class"].value_counts())

print("\nPercentage Distribution:")
print(df_ml["rating_class"].value_counts(normalize=True) * 100)

# visualizing the dataset
(important ,incase lo rating books are larger in amount ,it will imbalance our model training)

In [ ]:
plt.figure(figsize=(7, 5))

sns.countplot(data=df_ml, x="rating_class", order=["Low", "Medium", "High"])

plt.title("Distribution of Book Rating Classes")
plt.xlabel("Rating Class")
plt.ylabel("Number of Books")
plt.show()

### Observations

1. The dataset contains 1,000 books.
2. Low-rated books represent 42.2% of the dataset.
3. Medium-rated books represent 20.3%.
4. High-rated books represent 37.5%.
5. The Medium class has the fewest observations.

The classes are not equally distributed. Therefore, stratified train-test splitting will be used to preserve the class proportions. Precision, recall and F1-score will also be considered during model evaluation.

## 4. Train-Test Split

The dataset is divided into 80% training data and 20% testing data.

Stratified splitting is used to maintain the original distribution of rating classes in both sets.

A random state of 42 ensures reproducibility.

 ### 4.1 Feature and Target Selection

The target variable is `rating_class`, which contains three categories: Low, Medium, and High.

The input features selected initially are:

 Price
 Stock quantity
 Category
 Description availability
 Log-transformed description length

Identifiers, constant columns, redundant price columns, and the original star rating are excluded.

The original star rating is excluded because it was used to create the classification target. Including it would cause target leakage and produce misleading model performance.

These five features form our initial feature set. Forward Feature Selection and Backward Feature Elimination will later be used to identify useful features.

In [ ]:
features = [
    "price",
    "stock_quantity",
    "category",
    "has_description",
    "description_length_log"
]

X = df_ml[features].copy()
y = df_ml["rating_class"].copy()

print("Input Features:")
print(X.head())

print("\nTarget:")
print(y.head())

print("\nFeature Shape:", X.shape)
print("Target Shape:", y.shape)

### 4.2 Train-Test Split

The dataset is divided into 80% training data and 20% testing data.

The `train_test_split()` function is used to perform this division.

Stratified sampling is applied because the three rating classes have different proportions. It maintains approximately the same class distribution in both training and testing datasets.

A random state of 42 is used to ensure reproducibility.

The training dataset is used for preprocessing, feature selection, and model training. The testing dataset is kept separate for final model evaluation.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training Features:", X_train.shape)
print("Testing Features:", X_test.shape)

print("\nTraining Target:", y_train.shape)
print("Testing Target:", y_test.shape)

In [ ]:
print("Training Class Distribution:")
print(y_train.value_counts(normalize=True) * 100)

print("\nTesting Class Distribution:")
print(y_test.value_counts(normalize=True) * 100)

### Observations

The dataset was successfully divided into training and testing sets using an 80/20 split.

- The training dataset contains 800 books (80%).
- The testing dataset contains 200 books (20%).
- Stratified sampling maintained approximately the same proportions of Low, Medium, and High rating classes in both datasets.
- The class distribution in the training and testing sets is close to the original distribution.

Using stratification helps ensure that all three rating categories are adequately represented during model training and evaluation.

The training dataset will be used for preprocessing, feature selection, and model training, while the testing dataset will remain separate for final evaluation.

### 5. Feature Preprocessing

Before applying classification algorithms, the input features must be converted into a suitable numerical format.

The dataset contains numerical, categorical, and boolean features. We will use One-Hot Encoding to convert categorical features into numerical columns and StandardScaler to standardize numerical features.

Preprocessing will be fitted on the training data and then applied to the testing data to prevent data leakage.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_features = [
    "price",
    "stock_quantity",
    "description_length_log",
    "has_description"
]

categorical_features = ["category"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

print("Training data shape:", X_train_processed.shape)
print("Testing data shape:", X_test_processed.shape)

### Observations

The preprocessing step was completed successfully.

- The training dataset contains 800 samples and 51 features.
- The testing dataset contains 200 samples and 51 features.
- StandardScaler was applied to numerical features.
- One-Hot Encoding converted the categorical category feature into numerical columns.
- The same preprocessing rules were applied to both training and testing datasets.

The resulting datasets are now in numerical format and ready for feature selection and classification model training.

### 6. Feature Selection

Feature selection helps identify the most relevant input features for a machine learning task. It can reduce unnecessary features, simplify the model, and potentially improve performance.

We will apply Forward Feature Selection and Backward Feature Elimination using Logistic Regression as the estimator. Macro F1-score will be used to evaluate feature combinations because the target contains three rating classes with different sample counts.

In [ ]:
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold

# Get the names of all preprocessed features
feature_names = preprocessor.get_feature_names_out()

# Create the classification estimator
estimator = LogisticRegression(max_iter=2000)

# Use 3-fold stratified cross-validation
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

print("Total available features:", len(feature_names))
print("Feature names:")
print(feature_names)